In [0]:
import re
from datetime import datetime
from abc import ABC, abstractmethod

class BaseExtractor(ABC):
    @abstractmethod
    def extract(self, text: str) -> dict:
        pass

class CoyoteExtractor(BaseExtractor):
    def _normalize(self, text: str) -> str:
        if not text:
            return ""
        text = re.sub(r"\*{2,}", "", text)
        text = re.sub(r"[ \t\u00A0]+", " ", text)
        text = re.sub(r"\r\n?", "\n", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    def _extract_all_stops(self, raw_text: str) -> list:
        stop_pattern = r'(?im)^(?:#{1,3}\s*)?(Stop\s+\d+\s*:\s*(?:Pick\s*Up|Delivery))'
        parts = re.split(stop_pattern, raw_text)
        stops = []
        i = 1
        while i < len(parts) - 1:
            header = parts[i]
            body   = parts[i + 1] if (i + 1) < len(parts) else ""
            stops.append({"type": "pickup" if re.search(r'(?i)pick\s*up', header) else "delivery",
                          "number": int(re.search(r'(\d+)', header).group(1)) if re.search(r'(\d+)', header) else 0,
                          "text": header + "\n" + body})
            i += 2
        return stops

    def _extract_facility(self, block: str) -> str:
        m = re.search(
            r'Facility\s+(.+?)'
            r'(?=\s+(?:Address|SLIC|Contact|Phone|Notes|Facility|Numbers|No\s+Touch|Confirmation|>|Mon|Tue|Wed|Thu|Fri|Sat|Sun|\|))',
            block, re.I | re.S
        )
        if not m:
            return ""
        raw = m.group(1).strip()
        raw = re.sub(r'(?i)(Notes\s+Numbers.*|Notes\s+\-.*|Numbers\s+\d+.*)', '', raw).strip()
        raw = re.sub(r'~~\S+~~', '', raw)
        raw = re.sub(r'(?i)\s*>?\s*Confirmation\b.*', '', raw).strip()
        raw = re.sub(r'(?i)\s*No\s+Touch\b.*', '', raw).strip()
        raw = re.sub(r'(?i)\s*Driver\s+Work\b', ' ', raw).strip()
        raw = re.sub(r'\s+', ' ', raw)
        return raw.strip(" -|")

    def _extract_geo(self, block: str) -> dict:
        #
        result = {"address": "", "city": "", "state": "", "zipcode": ""}
        addr_zone_m = re.search(
            r'Address\s+(.+?)(?=\s+(?:Contact|Phone|Facility\s+Notes|STACEY|FRCO|$))',
            block, re.I | re.S
        )
        if not addr_zone_m:
            return result
        zone = addr_zone_m.group(1)
        zone = re.sub(r'(?i)\bSLIC\b', '', zone)
        zone = re.sub(r'(?i)\bN/A\b', '', zone)
        zone = re.sub(r'(?i)Driver\s+Work', '', zone)
        zone = re.sub(r'(?i)No\s+Touch', '', zone)
        zone = re.sub(r'(?i)Lumper', '', zone)
        zone = re.sub(r'~~\S+~~', '', zone)
        zone = " ".join(zone.split())
        print(f"DEBUG zone: [{zone}]")

        # Caso A: separar address de city usando SLIC o N/A como delimitador
        geo_m = re.search(r'(.+),\s*([A-Z]{2})\s+(\d{5}(?:-\d{4})?)\s*$', zone, re.I)
        if geo_m:
            before_comma = geo_m.group(1).strip()
            state   = geo_m.group(2).strip().upper()
            zipcode = geo_m.group(3).strip()

            # La ciudad es la última palabra antes de la coma
            # La dirección es todo antes del primer SLIC o N/A
            slic_m = re.search(r'^(.+?)\s+(?:SLIC|N/A)\b', before_comma, re.I)
            if slic_m:
                address = slic_m.group(1).strip().upper()
            else:
                # Sin SLIC ni N/A: ciudad es última palabra, dirección el resto
                words = before_comma.split()
                address = " ".join(words[:-1]).upper()

            # Ciudad es siempre la última palabra antes de la coma
            city = before_comma.split()[-1]

            result["address"] = address
            result["city"]    = city
            result["state"]   = state
            result["zipcode"] = zipcode
            return result
    


    def extract(self, text: str) -> dict:
        return {}

test_texts = {

    "28861101": """Stop 1: Pick Up ~~eee~~ 
    ## Pick Up PO-1162791 Numbers 
    Numbers Scheduled For Wed 03/29/2023 Confirmation None from 08:00 - 13:00 Numbers ~~|~~ Facility United Sugars Driver Work No Touch Address 450 SONORA DRIVE GATE D SLIC Clewiston, FL 33440 N/A 
    Contact None Phone +1 (863) 902 2707 
    ## Facility Notes 
    Food Grade Trailer Required
    ## Stop 2: Delivery ~~eee~~""",

        "29671154": """Stop 1: Pick Up ~~eee~~
    SLIC Address 220 GREENWOOD CT N/A SUITE 230 McDonough, GA 30253
    Contact STACEY CLEMENTS PIMS CONTACT Phone None
    ## Stop 2: Delivery ~~eee~~""",

        "30604868": """Stop 1: Pick Up ~~eee~~ 
    Numbers 169708130 Appointment Scheduled For Sun 01/07/2024 Confirmation None at 06:00 Numbers ~~|~~ Facility DSC-JM Smucker / Big Driver Work Heart Pet DC No Touch 
    Address 5000 BOHANNON DR SLIC Building A N/A Building A Fairburn, GA 30213 
    Contact FRCO@CJLOGISTICS AMERICA.COM
    ## Stop 2: Delivery ~~eee~~"""
    }

extractor = CoyoteExtractor()
for load_id, text in test_texts.items():
    print(f"\n{'='*60}")
    print(f"LOAD: {load_id}")
    normalized = extractor._normalize(text)
    stops = extractor._extract_all_stops(normalized)
    for stop in stops:
        if stop["type"] == "pickup":
            print(f"FACILITY: {extractor._extract_facility(stop['text'])}")
            print(f"GEO:      {extractor._extract_geo(stop['text'])}")

extractor = CoyoteExtractor()
text = extractor._normalize(test_text)
stops = extractor._extract_all_stops(text)

for stop in stops:
    print(f"\n--- STOP {stop['number']} ({stop['type']}) ---")
    print(f"BLOCK:\n{stop['text']}")
    print(f"FACILITY: {extractor._extract_facility(stop['text'])}")
    print(f"GEO: {extractor._extract_geo(stop['text'])}")